In [3]:
import os
os.chdir("../")

In [6]:
import os

# Set working directory to the root of your project
os.chdir("c:/Users/Babar/Documents/Chatbot/clean-cardiobot")

# Confirm the working directory
print("✅ Current working directory:", os.getcwd())

# Ensure research folder exists
os.makedirs("research", exist_ok=True)


✅ Current working directory: c:\Users\Babar\Documents\Chatbot\clean-cardiobot


In [1]:
# 📦 Essential imports
import os
import re
import json
from tqdm import tqdm
from datetime import datetime
from dotenv import load_dotenv

from langchain.schema import Document
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import Pinecone as LangchainPinecone
from langchain.chat_models import ChatOpenAI
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains import LLMChain


In [6]:

# Load PDFs from a folder
def load_pdf_file(Data):
    loader = DirectoryLoader(Data, glob="*.pdf", loader_cls=PyPDFLoader)
    return loader.load()

# Clean noisy patterns from raw PDF text
def clean_text(text):
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'(Page \d+|Figure \d+|Table \d+)', '', text, flags=re.IGNORECASE)
    text = re.sub(r'https?:\/\/\S+|www\.\S+', '', text)
    text = re.sub(r'^\s*\d+\s*$', '', text, flags=re.MULTILINE)
    text = re.sub(r'(Copyright|Elsevier|Permissions|ISBN|Editor.*?Edition)', '', text, flags=re.IGNORECASE)
    return text.strip()

# Clean all loaded documents
def clean_documents(docs):
    cleaned = []
    for doc in docs:
        content = clean_text(doc.page_content)
        if len(content.strip()) > 50:
            cleaned.append(Document(page_content=content, metadata=doc.metadata))
    return cleaned

# Split documents into smaller chunks for embedding
def split_documents(docs, chunk_size=1000, chunk_overlap=150):
    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    return splitter.split_documents(docs)

# Load, clean, and split all at once
raw_docs = load_pdf_file("Data/")
cleaned_docs = clean_documents(raw_docs)
text_chunks = split_documents(cleaned_docs)

print(f"✅ Loaded {len(raw_docs)} raw docs")
print(f"✅ Cleaned {len(cleaned_docs)} docs")
print(f"✅ Split into {len(text_chunks)} chunks")


Could not reliably determine page label for 128.
Could not reliably determine page label for 256.
Could not reliably determine page label for 312.
Could not reliably determine page label for 313.
Could not reliably determine page label for 314.
Could not reliably determine page label for 315.
Could not reliably determine page label for 316.
Could not reliably determine page label for 317.
Could not reliably determine page label for 318.
Could not reliably determine page label for 319.
Could not reliably determine page label for 320.
Could not reliably determine page label for 321.
Could not reliably determine page label for 128.
Could not reliably determine page label for 256.
Could not reliably determine page label for 312.
Could not reliably determine page label for 313.
Could not reliably determine page label for 314.
Could not reliably determine page label for 315.
Could not reliably determine page label for 316.
Could not reliably determine page label for 317.
Could not reliably d

✅ Loaded 19977 raw docs
✅ Cleaned 19016 docs
✅ Split into 95277 chunks


In [2]:
# ✅ Load HuggingFace BioBERT Embeddings
from langchain.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="pritamdeka/BioBERT-mnli-snli-scinli-scitail-mednli-stsb"
)
print("✅ BioBERT embeddings model loaded")

# ✅ Setup Pinecone connection
import os
from dotenv import load_dotenv
from pinecone.grpc import PineconeGRPC as Pinecone
from pinecone import ServerlessSpec

load_dotenv()

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# Connectting to Pinecone using existing index
pc = Pinecone(api_key=PINECONE_API_KEY)
index_name = "cardicbot"  # Reusing already populated index

from langchain_pinecone import PineconeVectorStore

vectorstore = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embeddings
)

print(f"✅ Connected to existing Pinecone index: '{index_name}'")


C:\Users\Babar\AppData\Local\Temp\ipykernel_7216\833675272.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
c:\Users\Babar\anaconda3\envs\cardiobot\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ BioBERT embeddings model loaded
✅ Connected to existing Pinecone index: 'cardicbot'


In [3]:
import os
from langchain_openai import ChatOpenAI
from openai import OpenAIError
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain

# ✅ Step 1: RAG system prompt
system_prompt = (
    "You are a helpful and knowledgeable assistant specialized in cardiology and cardiovascular medicine. "
    "Use only the provided context from medical guidelines and textbooks to answer the question. "
    "When relevant, mention the document source name and page number in parentheses. "
    "If the answer is not contained in the context, reply with: 'I’m not sure based on the provided information.' "
    "Keep your answer medically accurate, concise (min 5 sentences and max 8-10 sentences if answer doesn't get completed in 5 sentences), and avoid speculation.\n\nContext:\n{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")
])

# ✅ Step 2: Check OpenAI API key and test LLM
openai_key = os.getenv("OPENAI_API_KEY")
assert openai_key is not None and len(openai_key) > 10, "❌ OPENAI_API_KEY is missing or too short!"

try:
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    _ = llm.invoke("Say hello")  # Test call
    print("✅ OpenAI GPT-4o-mini model loaded and API key is valid")
except Exception as e:
    raise ValueError(f"❌ OpenAI API key might be invalid or expired:\n{str(e)}")

# ✅ Step 3: LangChain document combination (stuff strategy)
question_answer_chain = create_stuff_documents_chain(
    llm=llm,
    prompt=prompt
)

# ✅ Step 4: Create retriever from Pinecone index
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 6})

# ✅ Step 5: Final RAG chain (retriever + QA chain)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

print("✅ RAG pipeline is ready")


✅ OpenAI GPT-4o-mini model loaded and API key is valid
✅ RAG pipeline is ready


In [4]:
from datetime import datetime

def ask_query_and_log(query, rag_chain, retriever, log_to_file=True, debug=False):
    """
    Run a query through the RAG pipeline and optionally log and debug it.

    Parameters:
    - query (str): The user question.
    - rag_chain (Runnable): The LangChain RAG chain object (retriever + QA).
    - retriever (BaseRetriever): The retriever to get relevant documents.
    - log_to_file (bool): If True, logs output and context to rag_audit_log.txt.
    - debug (bool): If True, prints retrieved chunks (source, page, preview) to console.
                    Useful for verifying which documents were used to generate the answer.

    Returns:
    - str: The final formatted answer + sources (also printed to console).
    """
    response = rag_chain.invoke({"input": query})
    answer = response["answer"]
    documents = response["context"]

    sources = []
    for doc in documents:
        source_file = doc.metadata.get("source", "Unknown Source").split("\\")[-1]
        page_number = int(doc.metadata.get("page", 0))
        sources.append(f"{source_file}, page {page_number}")

    formatted_sources = "\n".join([f"- {s}" for s in sources])
    final_output = f"Query: {query}\n\nAnswer:\n{answer}\n\nSources:\n{formatted_sources}"

    # Print to console
    print("\n" + final_output)

    # Build debug preview if requested
    preview_log = ""
    if debug:
        preview_log += f"\n======================\nRetrieved Chunks for: '{query}'\n======================\n"
        retrieved_docs = retriever.get_relevant_documents(query)
        for doc in retrieved_docs:
            source = doc.metadata.get("source", "Unknown Source").split("\\")[-1]
            page = doc.metadata.get("page", "Unknown Page")
            content = doc.page_content.strip()[:300].replace("\n", " ")
            preview_log += f"\n---\nSource: {source}, Page: {page}\nContent Preview:\n{content}\n"
        print(preview_log)

    # Optionally log to file
    if log_to_file:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        with open("research/rag_audit_log.txt", "a", encoding="utf-8") as f:
            f.write(f"\n\n{timestamp}\n{final_output}\n{preview_log}\n")

    return final_output


In [7]:
ask_query_and_log("What is the role of beta blockers in managing heart failure?", rag_chain, retriever, debug=True)



Query: What is the role of beta blockers in managing heart failure?

Answer:
Beta blockers play a crucial role in managing heart failure, particularly in patients with left ventricular dysfunction. They have been unequivocally shown to reduce cardiac and sudden death mortality across various patient populations. In patients at risk of serious ventricular arrhythmias, especially those with left ventricular dysfunction, beta-blocker therapy is recommended unless contraindicated. 

While beta blockers can initially cause a short-term deterioration in cardiac function due to their negative inotropic effects, they ultimately lead to long-term benefits when used in conjunction with ACE inhibitors. These benefits include a decrease in left ventricular (LV) volumes, favorable changes in LV shape, and improved left ventricular ejection fraction (LVEF). Additionally, beta blockers improve patient symptoms, prevent hospitalizations, and enhance overall functional capacity in heart failure patien

'Query: What is the role of beta blockers in managing heart failure?\n\nAnswer:\nBeta blockers play a crucial role in managing heart failure, particularly in patients with left ventricular dysfunction. They have been unequivocally shown to reduce cardiac and sudden death mortality across various patient populations. In patients at risk of serious ventricular arrhythmias, especially those with left ventricular dysfunction, beta-blocker therapy is recommended unless contraindicated. \n\nWhile beta blockers can initially cause a short-term deterioration in cardiac function due to their negative inotropic effects, they ultimately lead to long-term benefits when used in conjunction with ACE inhibitors. These benefits include a decrease in left ventricular (LV) volumes, favorable changes in LV shape, and improved left ventricular ejection fraction (LVEF). Additionally, beta blockers improve patient symptoms, prevent hospitalizations, and enhance overall functional capacity in heart failure p

In [ ]:
from langchain.prompts import PromptTemplate
# ⛳️ Configuration
ENABLE_REFINEMENT = False                             # Optionally we can turn it off too
ALL_LOGS_FILE = "research/comparison_results.json"

# 🧠 LLM judge setup
EVAL_PROMPT_TEMPLATE = """
You are a medical cardio expert evaluating an assistant's answer to a clinical query. Use only the provided context.

Rate the answer from 1 to 5 (5 = excellent) on:

- Relevance
- Factual Accuracy
- Completeness
- Source Attribution
- Clarity

Then provide a one-line explanation and the average score.

---
Query: {query}

Context:
{context}

Answer:
{answer}

Evaluation Format:
Relevance: X  
Factual Accuracy: X  
Completeness: X  
Source Attribution: X  
Clarity: X  
Explanation: <your explanation>  
Average Score: X
"""

judge_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def evaluate_answer(query, context_docs, answer):
    context_text = "\n\n".join([doc.page_content[:300] for doc in context_docs])
    eval_prompt = PromptTemplate.from_template(EVAL_PROMPT_TEMPLATE)
    prompt_input = eval_prompt.format(query=query, context=context_text, answer=answer)
    return judge_llm.invoke(prompt_input).content

def extract_score(text):
    scores = {}
    pattern = r"(Relevance|Factual Accuracy|Completeness|Source Attribution|Clarity):\s*([0-5](\.\d+)?)"
    for match in re.findall(pattern, text, re.IGNORECASE):
        scores[match[0].strip()] = float(match[1])

    avg_match = re.search(r"Average Score:\s*([0-5](\.\d+)?)", text)
    avg_score = float(avg_match.group(1)) if avg_match else None
    return scores, avg_score

# 🛠 Refinement setup
refinement_prompt = PromptTemplate.from_template("""
You are a senior medical editor. Improve the assistant's answer to the medical query below.

Focus on improving the following area(s) based on expert LLM feedback: {weak_dimensions}.

Ensure your answer is accurate, concise (≤8 sentences), context-based, and includes inline citations where appropriate. Do not add hallucinated content.

Query: {query}
Original Answer: {answer}

Sources:
{sources}

Refined Answer:
""")

refinement_chain = LLMChain(llm=llm, prompt=refinement_prompt)

def refine_answer_smart(query, answer, sources, weak_dimensions):
    weak_str = ", ".join(weak_dimensions)
    source_text = "\n".join([f"- {s['source']}, page {s['page']}" for s in sources])
    return refinement_chain.invoke({
        "query": query,
        "answer": answer,
        "sources": source_text,
        "weak_dimensions": weak_str
    })["text"]

# 🔁 Comparison loop
def compare_k_results_with_judge(query, k_values=[4]):       # We can reduce it too
    local_log = []
    best_result = None
    highest_score = (-1, -1)  # avg, completeness

    for k in k_values:
        print(f"\n====================\nRunning with k = {k}\n====================")
        retrieved_docs = retriever.invoke(query, config={"k": k})

        # Deduplicate
        seen = set()
        unique_docs = []
        for doc in retrieved_docs:
            key = (doc.metadata.get("source"), doc.metadata.get("page"))
            if key not in seen:
                seen.add(key)
                unique_docs.append(doc)

        # Get answer
        input_payload = {"input": query, "context": unique_docs}
        result = question_answer_chain.invoke(input_payload)

        # Prepare source metadata
        sources = [{
            "source": doc.metadata.get("source", "Unknown").split("\\")[-1],
            "page": int(doc.metadata.get("page", 0))
        } for doc in unique_docs]

        # Judge
        evaluation = evaluate_answer(query, unique_docs, result)
        scores, avg = extract_score(evaluation)
        score_tuple = (avg, scores.get("Completeness", 0))

        print(f"\n📊 Evaluation for k={k}")
        print("Answer:\n", result)
        print("LLM Judge Feedback:\n", evaluation)
        print(f"📈 Scores: {scores}")
        print(f"🏁 Avg Score: {avg}")

        # Track best
        if score_tuple > highest_score:
            highest_score = score_tuple
            best_result = {
                "k": k,
                "answer": result,
                "score": avg,
                "dimension_scores": scores,
                "sources": sources
            }

        local_log.append({
            "query": query,
            "k": k,
            "answer": result,
            "llm_judge_feedback": evaluation,
            "llm_judge_score": {
                "dimension_scores": scores,
                "average": avg
            },
            "sources": sources,
            "refined_answer": None
        })

    # Refinement
    weak_areas = [k for k, v in best_result["dimension_scores"].items() if v < 5.0]
    refined_answer = None
    if ENABLE_REFINEMENT and weak_areas:
        print(f"\n🔧 Refining answer based on weaknesses: {weak_areas}...")
        refined_answer = refine_answer_smart(
            query, best_result["answer"], best_result["sources"], weak_areas
        )
        print("\n🧪 Refined Answer:\n", refined_answer)
    elif ENABLE_REFINEMENT:
        print("\n✅ No refinement needed. All scores are 5.0.")

    # Save refined result
    for entry in local_log:
        if entry["k"] == best_result["k"]:
            entry["refined_answer"] = refined_answer

    # Persist to file
    all_logs = []
    if os.path.exists(ALL_LOGS_FILE):
        with open(ALL_LOGS_FILE, "r", encoding="utf-8") as f:
            all_logs = json.load(f)

    all_logs.extend(local_log)
    with open(ALL_LOGS_FILE, "w", encoding="utf-8") as f:
        json.dump(all_logs, f, indent=2)

    print(f"\n🏆 Best k based on judge score: {best_result['k']} (Score: {best_result['score']})")
    print("\n📋 Final Best Answer:\n", best_result["answer"])
    if refined_answer:
        print("\n🪄 Refined Answer:\n", refined_answer)

    return best_result


C:\Users\Babar\AppData\Local\Temp\ipykernel_7216\4231410169.py:74: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  refinement_chain = LLMChain(llm=llm, prompt=refinement_prompt)


In [10]:
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [19]:
# Extracting data from pdf 
def load_pdf_file(Data):
    loader= DirectoryLoader(Data,
                            glob="*.pdf",
                            loader_cls=PyPDFLoader)
    
    documents= loader.load()
    return documents

In [ ]:
extracted_data = load_pdf_file(Data='Data/')

In [ ]:
extracted_data

In [ ]:
#Clean page contents
import re
def clean_text(text):
    text = re.sub(r'\s+', ' ', text)  # normalize whitespace
    text = re.sub(r'(Page \d+|Figure \d+|Table \d+)', '', text, flags=re.IGNORECASE)
    text = re.sub(r'https?:\/\/\S+|www\.\S+', '', text)  # removing URLs
    text = re.sub(r'^\s*\d+\s*$', '', text, flags=re.MULTILINE)  # removing numeric-only lines
    text = re.sub(r'(Copyright|Elsevier|Permissions|ISBN|Editor.*?Edition)', '', text, flags=re.IGNORECASE)
    return text.strip()

In [46]:
# Applying cleaning to all docs
from langchain.schema import Document

def clean_documents(docs):
    cleaned = []
    for doc in docs:
        content = clean_text(doc.page_content)
        if len(content.strip()) > 50:  # skip empty/short content
            cleaned.append(Document(page_content=content, metadata=doc.metadata))
    return cleaned

In [ ]:
# Cleaning extracted_data
cleaned_docs = clean_documents(extracted_data)
print(f"✅ Cleaned docs: {len(cleaned_docs)}")

In [48]:
# Splitting data into chunks

def text_split(docs):
    text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
    text_chunks=text_splitter.split_documents(docs)
    return text_chunks


In [ ]:
text_chunks=text_split(cleaned_docs)
print("Length of text chunks", len(text_chunks))

In [50]:
from langchain.embeddings import HuggingFaceEmbeddings

In [51]:
# Downloading Embedding Model

def download_hugging_face_embeddings():
    embeddings=HuggingFaceEmbeddings(model_name= "pritamdeka/BioBERT-mnli-snli-scinli-scitail-mednli-stsb"
)
    return embeddings

In [52]:
embeddings = download_hugging_face_embeddings()

In [ ]:
Query_result = embeddings.embed_query("Hello World")
print("Length", len(Query_result))

In [ ]:
Query_result

In [63]:
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
PINECONE_API_KEY = os.environ.get('PINECONE_API_KEY')
OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY')

In [ ]:
from pinecone.grpc import PineconeGRPC as Pinecone
from pinecone import ServerlessSpec
import os

pc = Pinecone(api_key = os.getenv("PINECONE_API_KEY"))

index_name = "cardicbot"
dimension = 768

if index_name in [index.name for index in pc.list_indexes().indexes]:
    pc.delete_index(index_name)
    print("🧹 Old index deleted")

pc.create_index(
    name=index_name,
    dimension=768, # Replace with your model dimensions
    metric="cosine", # Replace with your model metric
    spec=ServerlessSpec(
        cloud="aws",
        region="us-east-1"
    ) 
)

In [65]:
import os
os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

In [ ]:
# Embeddings into Pinecone index

from langchain_pinecone import  PineconeVectorStore
from tqdm import tqdm
def batch_upload_to_pinecone(text_chunks, embeddings, index_name, batch_size=100):

    texts = [doc.page_content for doc in text_chunks]
    metadatas = [doc.metadata for doc in text_chunks]

    for i in tqdm(range(0, len(texts), batch_size)):
        batch_texts = texts[i:i+batch_size]
        batch_metas = metadatas[i:i+batch_size]

        try:
            PineconeVectorStore.from_texts(
                texts=batch_texts,
                embedding=embeddings,
                metadatas=batch_metas,
                index_name=index_name,
            )

        except Exception as e:
            print(f"Batch {i}-{i+batch_size} failed: {e}")    
 
batch_upload_to_pinecone(text_chunks, embeddings, index_name)

In [62]:
from langchain_pinecone import PineconeVectorStore
from tqdm import tqdm
import time
import os
import json

def batch_upload_to_pinecone_range(text_chunks, embeddings, index_name, batch_size=100, start_batch=0, end_batch=None, log_file="upload_log.json"):
    index = PineconeVectorStore.get_pinecone_index(index_name=index_name)

    texts = [doc.page_content for doc in text_chunks]
    metadatas = [doc.metadata for doc in text_chunks]

    total_batches = len(texts) // batch_size + 1
    if end_batch is None or end_batch > total_batches:
        end_batch = total_batches

    # Load already completed batch logs
    if os.path.exists(log_file):
        with open(log_file, "r") as f:
            completed = set(json.load(f))
    else:
        completed = set()

    for batch_num in tqdm(range(start_batch, end_batch), desc=f"Uploading batches {start_batch} to {end_batch}"):
        if batch_num in completed:
            continue

        start = batch_num * batch_size
        end = min(start + batch_size, len(texts))

        batch_texts = texts[start:end]
        batch_metas = metadatas[start:end]

        try:
            PineconeVectorStore.from_texts(
                texts=batch_texts,
                embedding=embeddings,
                metadatas=batch_metas,
                index_name=index_name,
            )
            # Log successful batch
            completed.add(batch_num)
            with open(log_file, "w") as f:
                json.dump(list(completed), f)

        except Exception as e:
            print(f"❌ Batch {batch_num} failed: {e}")
            time.sleep(5)
            continue


In [ ]:
from pinecone.grpc import PineconeGRPC as Pinecone
from pinecone import ServerlessSpec
import os
from dotenv import load_dotenv

# Load API key from .env
load_dotenv()
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

# Access your index
index = pc.Index("cardicbot")

# Print stats
print(index.describe_index_stats())


In [ ]:
import os
from dotenv import load_dotenv
from pinecone.grpc import PineconeGRPC as Pinecone
from pinecone import ServerlessSpec

load_dotenv()
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))


In [2]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="pritamdeka/BioBERT-mnli-snli-scinli-scitail-mednli-stsb"
)


In [3]:
from langchain_pinecone import PineconeVectorStore

index_name = "cardicbot"  

vectorstore = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embeddings
)


In [ ]:
query = "Symptoms of acute coronary syndrome"
docs = vectorstore.similarity_search(query, k=3)

for i, doc in enumerate(docs):
    print(f"\n📄 Result {i+1}:\n{doc.page_content[:500]}...\n")


In [ ]:
queries = [
    "Symptoms of acute coronary syndrome",
    "Management of NSTEMI",
    "Dual antiplatelet therapy duration",
    "Role of troponins in ACS",
    "High-risk features in unstable angina",
    "Contraindications to fibrinolysis",
    "STEMI complications",
    "What are the signs of a heart attack?",
    "Why is chest pain important in heart patients?"
]

for query in queries:
    print(f"\n🔍 Query: {query}")
    docs = vectorstore.similarity_search(query, k=2)  # change k if needed
    
    for i, doc in enumerate(docs):
        print(f"\n📄 Result {i+1}:\n{doc.page_content[:500]}...\n")
    print("=" * 80)


In [96]:
retriever = vectorstore.as_retriever(search_type = "similarity", search_kwargs={"k":6})

In [67]:
retrieved_docs = retriever.invoke("What are Symptoms of acute coronary syndrome?")

In [ ]:
retrieved_docs

In [ ]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model_name="gpt-4o-mini", temperature=0.3, key="OPENAI_API_KEY")


In [ ]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

system_prompt = (
    "You are a helpful and knowledgeable assistant specialized in cardiology and cardiovascular medicine."
    " Use only the provided context from medical guidelines and textbooks to answer the question."
    " When relevant, mention the document source name and page number in parentheses."
    " If the answer is not contained in the context, reply with: 'I’m not sure based on the provided information.'"
    " Keep your answer medically accurate, concise (max 5 sentences), and avoid speculation."
    "\n\nContext:\n{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

In [97]:
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

In [ ]:
response = rag_chain.invoke({"input": "Describe the role of troponin in diagnosing myocardial infarction."})

# Extracting answer
answer = response["answer"]

sources = []
for doc in response["context"]:
    source_file = doc.metadata.get("source", "Unknown Source").split("\\")[-1]
    page_number = int(doc.metadata.get("page", 0))
    sources.append(f"{source_file}, page {page_number}")

# Formatting the final response
formatted_sources = "\n".join([f"- {s}" for s in sources])
final_output = f"Answer:\n{answer}\n\nSources:\n{formatted_sources}"

print(final_output)

In [ ]:
from datetime import datetime

def ask_query_and_log(query, rag_chain, retriever, log_to_file=True):
    # Step 1: Run RAG pipeline
    response = rag_chain.invoke({"input": query})
    answer = response["answer"]
    documents = response["context"]

    # Step 2: Extract and format sources
    sources = []
    for doc in documents:
        source_file = doc.metadata.get("source", "Unknown Source").split("\\")[-1]
        page_number = int(doc.metadata.get("page", 0))
        sources.append(f"{source_file}, page {page_number}")

    # Step 3: Build formatted output
    formatted_sources = "\n".join([f"- {s}" for s in sources])
    final_output = f"Query: {query}\n\nAnswer:\n{answer}\n\nSources:\n{formatted_sources}"

    # Step 4: Log each retrieved chunk with metadata
    retrieved_docs = retriever.get_relevant_documents(query)
    debug_log = f"\n======================\nRetrieved Chunks for: '{query}'\n======================\n"
    for doc in retrieved_docs:
        source = doc.metadata.get("source", "Unknown Source").split("\\")[-1]
        page = doc.metadata.get("page", "Unknown Page")
        content = doc.page_content.strip()[:300].replace("\n", " ")  # Short preview
        debug_log += f"\n---\nSource: {source}, Page: {page}\nContent Preview:\n{content}\n"

    # Step 5: Print output to console
    print("\n" + final_output)
    print(debug_log)

    # Step 6: Optional - Save to file
    if log_to_file:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        with open("rag_audit_log.txt", "a", encoding="utf-8") as f:
            f.write(f"\n\n{timestamp}\n" + final_output + "\n" + debug_log + "\n")

# Example queries
queries = [
    "What is Acute coronary syndrome?",
    "How is heart failure with preserved ejection fraction (HFpEF) managed?",
    "What are the common risk factors for developing coronary artery disease?"
]

# Run for all queries
for q in queries:
    ask_query_and_log(q, rag_chain, retriever)


In [ ]:
queries = [
    "What is the recommended initial management for STEMI?",
    "How is atrial fibrillation managed in patients with heart failure?",
    "What are the contraindications to beta-blockers in cardiovascular patients?",
    "What are the current ACC/AHA recommendations for statin therapy?",
    "What ECG changes are seen in hyperkalemia?",
    "Differentiate between NSTEMI and unstable angina."

]

# Run the queries
for q in queries:
    ask_query_and_log(q, rag_chain, retriever)

FIRST VERSION

In [ ]:

import json
import os
import re

# File will store all comparison question/ queries asked
ALL_LOGS_FILE = "comparison_results.json"

# To clean filenames
def sanitize_filename(name):
    return re.sub(r'[\\/*?:"<>|]', "_", name)

def compare_k_results(query, k_values=[4, 6, 8, 10]):
    # Local log to collect results for this query
    local_log = []

    for k in k_values:
        print(f"\n====================\nRunning with k = {k}\n====================")
        
        # Step 1: Retrieving the documents
        retrieved_docs = retriever.get_relevant_documents(query, k=k)

        # Step 2: Deduplicate by source, page
        seen = set()
        unique_docs = []
        for doc in retrieved_docs:
            key = (doc.metadata.get("source"), doc.metadata.get("page"))
            if key not in seen:
                seen.add(key)
                unique_docs.append(doc)

        # Step 3: Building input chain
        input_payload = {
            "input": query,
            "context": unique_docs
        }

        # Step 4: Run chain
        result = question_answer_chain.invoke(input_payload)

        # Step 5: Extracting source information
        sources = []
        for doc in unique_docs:
            source_file = doc.metadata.get("source", "Unknown").split("\\")[-1]
            page_number = int(doc.metadata.get("page", 0))
            print(f"- {source_file}, page {page_number}")
            sources.append({
                "source": source_file,
                "page": page_number
            })

        print("\nAnswer:\n", result)

        # Step 6: Adding result to log
        local_log.append({
            "query": query,
            "k": k,
            "answer": result if isinstance(result, str) else str(result),
            "sources": sources
        })

    # Step 7: Append to master log file
    if os.path.exists(ALL_LOGS_FILE):
        with open(ALL_LOGS_FILE, "r", encoding="utf-8") as f:
            all_logs = json.load(f)
    else:
        all_logs = []

    all_logs.extend(local_log)

    with open(ALL_LOGS_FILE, "w", encoding="utf-8") as f:
        json.dump(all_logs, f, indent=2)

    print(f"\n✅ All results for '{query}' saved in: {ALL_LOGS_FILE}")


LATEST VERSION

In [ ]:
import json
import os
import re
from langchain.prompts import PromptTemplate
from langchain.chat_models import ChatOpenAI

ENABLE_REFINEMENT = True  # Refining in batch mode

# LLM judge setup
EVAL_PROMPT_TEMPLATE = """
You are a medical expert evaluating an assistant's answer to a clinical query. Use only the provided context.

Rate the answer from 1 to 5 (5 = excellent) on:

- Relevance
- Factual Accuracy
- Completeness
- Source Attribution
- Clarity

Then provide a one-paragraph explanation and the average score.

---
Query: {query}

Context:
{context}

Answer:
{answer}

Evaluation Format:
Relevance: X  
Factual Accuracy: X  
Completeness: X  
Source Attribution: X  
Clarity: X  
Explanation: <your explanation>  
Average Score: X
"""


judge_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, key="OPENAI_API_KEY")

def evaluate_answer(query, context_docs, answer):
    context_text = "\n\n".join([doc.page_content[:300] for doc in context_docs])
    eval_prompt = PromptTemplate.from_template(EVAL_PROMPT_TEMPLATE)
    prompt_input = eval_prompt.format(query=query, context=context_text, answer=answer)
    return judge_llm.invoke(prompt_input).content

def extract_score(text):
    scores = {}
    pattern = r"(Relevance|Factual Accuracy|Completeness|Source Attribution|Clarity):\s*([0-5](\.\d+)?)"
    for match in re.findall(pattern, text, re.IGNORECASE):
        dimension = match[0].strip()
        score = float(match[1])
        scores[dimension] = score

    avg_match = re.search(r"Average Score:\s*([0-5](\.\d+)?)", text, re.IGNORECASE)
    avg = float(avg_match.group(1)) if avg_match else None
    return scores, avg

ALL_LOGS_FILE = "comparison_results.json"

from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
refinement_prompt = PromptTemplate.from_template("""
You are a senior medical editor. Improve the assistant's answer to the medical query below.

Focus on improving the following area(s) based on expert LLM feedback: {weak_dimensions}.

Ensure your answer is accurate, concise (≤5 sentences), context-based, and includes inline citations where appropriate (e.g., Hurst's the Heart, p.495). Do not add hallucinated content.

Query: {query}
Original Answer: {answer}

Sources:
{sources}

Refined Answer:
""")

refinement_chain = LLMChain(llm=llm, prompt=refinement_prompt)
def refine_answer_smart(query, answer, sources, weak_dimensions):
    weak_string = ", ".join(weak_dimensions)
    source_text = "\n".join([f"- {s['source']}, page {s['page']}" for s in sources])
    return refinement_chain.invoke({
        "query": query,
        "answer": answer,
        "sources": source_text,
        "weak_dimensions": weak_string
    })["text"]


def compare_k_results_with_judge(query, k_values=[4, 6, 8, 10]):
    local_log = []
    best_result = None
    highest_score = (-1, -1)

    for k in k_values:
        print(f"\n====================\nRunning with k = {k}\n====================")
        retrieved_docs = retriever.get_relevant_documents(query, k=k)

        seen = set()
        unique_docs = []
        for doc in retrieved_docs:
            key = (doc.metadata.get("source"), doc.metadata.get("page"))
            if key not in seen:
                seen.add(key)
                unique_docs.append(doc)

        input_payload = {"input": query, "context": unique_docs}
        result = question_answer_chain.invoke(input_payload)

        sources = [{
            "source": doc.metadata.get("source", "Unknown").split("\\")[-1],
            "page": int(doc.metadata.get("page", 0))
        } for doc in unique_docs]

        evaluation = evaluate_answer(query, unique_docs, result)
        scores_dict, avg_score = extract_score(evaluation)
        completeness_score = scores_dict.get("Completeness", 0)
        score_tuple = (avg_score, completeness_score)

        print(f"\n📊 Evaluation for k={k}")
        print("Answer:\n", result)
        print("LLM Judge Feedback:\n", evaluation)
        print(f"📈 Scores: {scores_dict}")
        print(f"🏁 Avg Score: {avg_score}")

        refined_answer = None

        if score_tuple > highest_score:
            highest_score = score_tuple
            best_result = {
                "k": k,
                "answer": result,
                "score": avg_score,
                "dimension_scores": scores_dict,
                "sources": sources
            }    

        local_log.append({
            "query": query,
            "k": k,
            "answer": result,
            "llm_judge_feedback": evaluation,
            "llm_judge_score": {
                "dimension_scores": scores_dict,
                "average": avg_score
            },
            "sources": sources,
            "refined_answer": refined_answer
        })

    print(f"\n🏆 Best k based on judge score: {best_result['k']} (Score: {best_result['score']})")
    print("\n📋 Best Answer Before Refinement:\n", best_result["answer"])
    

    weak_areas = [k for k, v in best_result["dimension_scores"].items() if v < 5.0]
    refined_answer = None
    if ENABLE_REFINEMENT and refinement_chain and weak_areas:
        print(f"\n🔧 Refining answer based on weaknesses: {weak_areas}...")
        refined_answer = refine_answer_smart(
            query=query,
            answer=best_result["answer"],
            sources=best_result["sources"],
            weak_dimensions=weak_areas
        )
        print("\n🧪 Refined Answer:\n", refined_answer)
    elif ENABLE_REFINEMENT and refinement_chain:
        print("\n✅ No refinement needed. All scores are 5.0.")

    for entry in local_log:
        if entry["k"] == best_result["k"]:
            entry["refined_answer"] = refined_answer
            break


    if os.path.exists(ALL_LOGS_FILE):
        with open(ALL_LOGS_FILE, "r", encoding="utf-8") as f:
            all_logs = json.load(f)
    else:
        all_logs = []

    all_logs.extend(local_log)
    with open(ALL_LOGS_FILE, "w", encoding="utf-8") as f:
        json.dump(all_logs, f, indent=2)

    print(f"\n✅ All results for '{query}' saved in: {ALL_LOGS_FILE}")
    
    best_result["refined_answer"] = refined_answer

    return best_result  


In [ ]:
compare_k_results_with_judge("What are the treatment goals in heart failure with reduced ejection fraction?")


In [ ]:
compare_k_results_with_judge("What is the role of beta blockers in managing heart failure?")


In [ ]:
compare_k_results_with_judge("What are the diagnostic criteria for hypertrophic cardiomyopathy?")


In [ ]:
compare_k_results_with_judge("What is the recommended treatment for STEMI?")

In [ ]:
compare_k_results_with_judge("What is atrial fibrillation?")

In [ ]:
compare_k_results_with_judge("How does NSTEMI differ from unstable angina?")

In [ ]:
compare_k_results_with_judge("What are the contraindications to beta-blockers?")

In [ ]:
compare_k_results_with_judge("What are the risk factors for coronary artery disease?")